# Evaluación Parcial N°2 — Agente TCG Funcional
## ISY0101 Ingeniería de Soluciones con IA — Empresa Trade SPA

| IL | Descripción |
|---|---|
| **IL2.1** | Agente funcional con herramientas de consulta, escritura y razonamiento |
| **IL2.2** | Memoria de corto y largo plazo + recuperación de contexto semántico |
| **IL2.3** | Planificación y toma de decisiones adaptativas |
| **IL2.4** | Documentación técnica y orquestación de componentes |

---
## IL2.1 — Agente Funcional con Herramientas (IE1, IE2)

Se implementa un agente LangChain con **tres herramientas especializadas** para el contexto de Empresa Trade SPA:

| Herramienta | Tipo | Descripción |
|---|---|---|
| `consultar_precio_carta` | Consulta | Busca precios en la base de conocimiento TCG |
| `buscar_informacion_tcg` | Recuperación RAG | Búsqueda léxica sobre documentos TCG |
| `calcular_valor_coleccion` | Razonamiento | Estima el valor total de una colección |

**Framework:** LangChain `create_openai_tools_agent` + `AgentExecutor`

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(
    base_url=os.getenv('OPENAI_BASE_URL'),
    api_key=os.getenv('GITHUB_TOKEN'),
    model='gpt-4o-mini',
    temperature=0.3
)
print(f'Modelo: {llm.model_name}')
print('Entorno configurado correctamente')

In [ ]:
# Base de conocimiento TCG (heredada de EP1 - IL1.3)
knowledge_base = [
    "Charizard Holo Base Set 1999 sin grading: USD 500-2000. PSA 9: USD 10000-30000. PSA 10: USD 300000+.",
    "Pikachu Illustrator: carta mas rara de Pokemon, 39 copias conocidas. Valor: USD 500000 a 5000000.",
    "Sobres Pokemon Stellar Crown 2024: USD 5 por sobre, USD 150 caja 36 sobres. Terapagos ex ~USD 40.",
    "Sobres Yu-Gi-Oh Phantom Nightmare 2024: USD 4-5 por sobre, USD 90-110 caja 24 sobres.",
    "Blue-Eyes White Dragon LOB 1ra edicion sin grading: USD 500-3000. BGS 10: mas de USD 50000.",
    "Dark Magician LOB 1ra edicion buen estado: USD 200-800.",
    "Black Lotus Alpha Magic The Gathering PSA 10: mas de USD 500000. Copia jugada: USD 5000-50000.",
    "Sobres Magic Modern Horizons 3 2024: USD 7-8 por sobre.",
    "Para saber si carta es valiosa: verificar edicion, rareza, estado y demanda en TCGPlayer o Cardmarket.",
    "Riftbound lanzado 2024 basado en League of Legends. Mercado secundario no consolidado. Sobres ~USD 5.",
    "Grading PSA y Beckett BGS son los mas reconocidos. PSA 10 puede valer 5-10 veces mas. Costo USD 20-100.",
    "Venta cartas TCG en Chile: MercadoLibre, grupos Facebook TCG Chile, tiendas como Empresa Trade SPA.",
]

def recuperar_docs(consulta, top_k=3):
    palabras = consulta.lower().split()
    puntuados = [(sum(1 for p in palabras if p in d.lower()), d) for d in knowledge_base]
    puntuados = [(s, d) for s, d in puntuados if s > 0]
    puntuados.sort(reverse=True)
    return [d for _, d in puntuados[:top_k]]

print(f'Base de conocimiento: {len(knowledge_base)} documentos cargados')

In [ ]:
# ── HERRAMIENTA 1: Consulta de precio de carta específica ──────────────
@tool
def consultar_precio_carta(nombre_carta: str) -> str:
    """Consulta el precio estimado de una carta TCG específica en la base de conocimiento
    de Empresa Trade SPA. Usa esta herramienta cuando el usuario pregunte por el valor
    o precio de una carta en particular."""
    docs = recuperar_docs(nombre_carta)
    if not docs:
        return f'No tengo datos de precio para "{nombre_carta}" en la base de conocimiento.'
    return 'Informacion encontrada:\n' + '\n'.join(f'- {d}' for d in docs)

# ── HERRAMIENTA 2: Búsqueda RAG general ────────────────────────────────
@tool
def buscar_informacion_tcg(consulta: str) -> str:
    """Busca informacion general sobre TCG (cartas, sobres, grading, venta)
    usando recuperacion aumentada sobre la base de conocimiento interna.
    Usa cuando la consulta sea sobre TCG en general, no una carta específica."""
    docs = recuperar_docs(consulta)
    if not docs:
        return 'No hay informacion especifica en la base de conocimiento para esta consulta.'
    return 'Contexto recuperado:\n' + '\n'.join(f'- {d}' for d in docs)

# ── HERRAMIENTA 3: Cálculo de valor de colección ───────────────────────
@tool
def calcular_valor_coleccion(descripcion_coleccion: str) -> str:
    """Estima el valor total de una coleccion de cartas TCG basandose en los datos
    disponibles. El usuario describe sus cartas y esta herramienta calcula un rango
    de valor estimado consultando la base de conocimiento."""
    docs = recuperar_docs(descripcion_coleccion, top_k=5)
    contexto = '\n'.join(f'- {d}' for d in docs) if docs else 'Sin datos especificos.'
    return (
        f'Para estimar el valor de la coleccion descrita como: "{descripcion_coleccion}"\n'
        f'Datos de referencia disponibles:\n{contexto}\n'
        'Nota: Para valuacion precisa, se recomienda consultar TCGPlayer o Cardmarket con cada carta.'
    )

tools = [consultar_precio_carta, buscar_informacion_tcg, calcular_valor_coleccion]
print(f'{len(tools)} herramientas configuradas: {[t.name for t in tools]}')

In [ ]:
# Prompt del agente con system message especializado en TCG
agent_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'Eres el Agente TCG de Empresa Trade SPA, experto en Trading Card Games.\n'
     'Tienes acceso a herramientas para consultar precios, buscar informacion y calcular valores.\n'
     'Razona paso a paso antes de usar una herramienta. Siempre explica que herramienta usas y por que.\n'
     'Si una consulta no es sobre TCG, indícalo amablemente y ofrece redirigir la consulta.'),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

# Crear agente y executor
agent = create_openai_tools_agent(llm, tools, agent_prompt)
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    return_intermediate_steps=True
)
print('Agente TCG creado con AgentExecutor')
print(f'Max iteraciones: {executor.max_iterations}')

In [ ]:
# IE6 - Demostración de toma de decisiones: 3 escenarios distintos

test_queries = [
    'Cuanto vale un Charizard de la primera edicion?',        # → usa consultar_precio_carta
    'Explícame como funciona el grading de cartas con PSA',   # → usa buscar_informacion_tcg
    'Tengo un Charizard y un Blue-Eyes, cuanto vale mi coleccion?', # → usa calcular_valor_coleccion
]

for i, q in enumerate(test_queries, 1):
    print(f'\n{'='*60}')
    print(f'CONSULTA {i}: {q}')
    print('='*60)
    result = executor.invoke({'input': q, 'chat_history': []})
    print(f'\nRESPUESTA FINAL: {result["output"][:300]}')
    if result.get('intermediate_steps'):
        herramientas_usadas = [step[0].tool for step in result['intermediate_steps']]
        print(f'Herramienta(s) usada(s): {herramientas_usadas}')

---
## IL2.2 — Configuración de Memoria (IE3, IE4)

Se implementan dos niveles de memoria para asegurar continuidad en conversaciones prolongadas:

| Tipo | Implementación | Propósito |
|---|---|---|
| **Corto plazo** | `ConversationBufferWindowMemory(k=5)` | Retiene las últimas 5 interacciones |
| **Largo plazo / Semántico** | RAG sobre knowledge base TCG | Recupera contexto de dominio relevante |

### IE3 — Memoria de contenido
`ConversationBufferWindowMemory` almacena el historial reciente. La ventana deslizante de k=5 controla el uso de tokens manteniendo coherencia conversacional.

### IE4 — Recuperación de contexto semántico
El agente usa la herramienta `buscar_informacion_tcg` como capa de recuperación semántica, accediendo a la base de conocimiento TCG de Empresa Trade SPA para enriquecer sus respuestas.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.messages import HumanMessage, AIMessage

# IE3 - Memoria de corto plazo: ventana deslizante de 5 turnos
memoria_corto = ConversationBufferWindowMemory(
    memory_key='chat_history',
    return_messages=True,
    k=5  # recuerda los ultimos 5 turnos de conversacion
)

# Agente con memoria integrada
executor_con_memoria = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memoria_corto,
    verbose=False,
    max_iterations=5
)
print('Agente con memoria de corto plazo configurado')
print(f'Ventana de memoria: k={memoria_corto.k} turnos')

In [ ]:
# Demostración de continuidad en conversación prolongada (IE3)
conversacion = [
    'Hola! Me llamo Carlos y colecciono cartas de Pokemon',
    'Tengo un Charizard Holo de 1999, cuanto puede valer?',
    'Y si lo mando a gradear con PSA, cuanto mas valdria?',
    'Donde puedo venderlo en Chile?',
    'Gracias! Recuerdas como me llamo?',  # prueba de memoria
]

print('=== DEMOSTRACIÓN DE MEMORIA CONVERSACIONAL ===')
for i, mensaje in enumerate(conversacion, 1):
    print(f'\n[Turno {i}] Usuario: {mensaje}')
    resp = executor_con_memoria.invoke({'input': mensaje})
    print(f'[Turno {i}] Agente: {resp["output"][:250]}')

# Mostrar estado de la memoria
historial = memoria_corto.load_memory_variables({})
print(f'\nTurnos en memoria: {len(historial["chat_history"])} mensajes')

In [ ]:
# IE4 - Recuperación de contexto semántico
# El agente selecciona automáticamente qué documentos recuperar según el contexto

print('=== DEMOSTRACIÓN DE RECUPERACIÓN SEMÁNTICA ===')

# Resetear memoria para nueva sesión
memoria_semantica = ConversationBufferWindowMemory(
    memory_key='chat_history', return_messages=True, k=5
)
executor_semantico = AgentExecutor(
    agent=agent, tools=tools, memory=memoria_semantica,
    verbose=False, max_iterations=5
)

# Flujo: el agente recupera contexto semántico progresivamente
flujo_semantico = [
    'Estoy pensando en invertir en cartas TCG, por donde empiezo?',
    'Me interesan mas las cartas de Yu-Gi-Oh. Que cartas valiosas hay?',
    'Si las mando a gradear, como afecta eso al precio que mencionaste?',
]

for i, msg in enumerate(flujo_semantico, 1):
    print(f'\n[{i}] Usuario: {msg}')
    r = executor_semantico.invoke({'input': msg})
    pasos = r.get('intermediate_steps', [])
    if pasos:
        tools_usadas = [p[0].tool for p in pasos]
        print(f'    Recuperacion semantica via: {tools_usadas}')
    print(f'    Agente: {r["output"][:200]}')

---
## IL2.3 — Planificación y Toma de Decisiones (IE5, IE6)

### IE5 — Esquema de Planificación

El agente sigue un esquema **Plan → Execute** con 4 etapas secuenciadas:

```
1. CLASIFICAR  → Determinar tipo de consulta (precio / info / colección / fuera de dominio)
2. RECUPERAR   → Seleccionar herramienta apropiada y ejecutarla
3. SINTETIZAR  → Combinar contexto recuperado con razonamiento del LLM
4. RESPONDER   → Generar respuesta final con recomendación práctica
```

### IE6 — Toma de Decisiones Adaptativas
El agente ajusta su comportamiento según la consulta: selecciona la herramienta correcta, decide cuántas iteraciones necesita, y responde diferente ante consultas dentro vs. fuera del dominio TCG.

In [ ]:
# IE5 - Planificador explícito: el agente genera un plan antes de ejecutar

from langchain_core.prompts import PromptTemplate

planner_prompt = PromptTemplate.from_template(
    'Eres el planificador del Agente TCG de Empresa Trade SPA.\n'
    'Dado el siguiente objetivo, genera un plan de 3-5 pasos ordenados para resolverlo.\n'
    'Cada paso debe especificar: (1) accion a tomar, (2) herramienta a usar si aplica.\n'
    'Herramientas disponibles: consultar_precio_carta, buscar_informacion_tcg, calcular_valor_coleccion\n\n'
    'OBJETIVO: {objetivo}\n\n'
    'PLAN (formato: Paso N: descripcion [herramienta: nombre]):'
)

planner_chain = planner_prompt | llm

# Demostrar planificación para una tarea compleja
objetivo = 'Un cliente quiere saber si vale la pena invertir en una coleccion de cartas Charizard para venderlas en Chile'

print('OBJETIVO:', objetivo)
print('\nPLAN GENERADO POR EL AGENTE:')
print('-' * 55)
plan = planner_chain.invoke({'objetivo': objetivo})
print(plan.content)

In [ ]:
# IE5 - Ejecutar el plan paso a paso

def ejecutar_plan(objetivo):
    print(f'OBJETIVO: {objetivo}')
    print('\n[Fase 1] Generando plan...')
    plan_resp = planner_chain.invoke({'objetivo': objetivo})
    print(plan_resp.content)

    print('\n[Fase 2] Ejecutando con el agente...')
    mem = ConversationBufferWindowMemory(memory_key='chat_history', return_messages=True, k=5)
    exec_plan = AgentExecutor(agent=agent, tools=tools, memory=mem, verbose=False, max_iterations=5)
    resultado = exec_plan.invoke({'input': objetivo})

    print('\n[Fase 3] Resultado final:')
    print(resultado['output'])
    pasos = resultado.get('intermediate_steps', [])
    if pasos:
        print(f'\nHerramientas usadas en ejecucion: {[p[0].tool for p in pasos]}')
    return resultado

r = ejecutar_plan('Quiero vender mi coleccion de cartas Yu-Gi-Oh antiguas en Chile. Que me recomiendas?')

In [ ]:
# IE6 - Toma de decisiones adaptativas: 4 escenarios con condiciones distintas

escenarios = [
    {
        'descripcion': 'Consulta dentro del dominio - precio especifico',
        'input': 'Cuanto vale un Black Lotus de Magic The Gathering?'
    },
    {
        'descripcion': 'Consulta fuera del dominio - debe declinar',
        'input': 'Cual es el precio del dolar hoy?'
    },
    {
        'descripcion': 'Consulta compleja - multiples herramientas',
        'input': 'Tengo Charizard, Blue-Eyes y Black Lotus. Cuanto vale todo y donde vendo en Chile?'
    },
    {
        'descripcion': 'Consulta sobre nuevo TCG - limite de conocimiento',
        'input': 'Cuanto vale una carta legendaria de Riftbound?'
    },
]

print('=== DEMOSTRACIÓN DE TOMA DE DECISIONES ADAPTATIVAS ===')
for sc in escenarios:
    print(f'\n[ESCENARIO] {sc["descripcion"]}')
    print(f'Input: {sc["input"]}')
    mem_sc = ConversationBufferWindowMemory(memory_key='chat_history', return_messages=True, k=3)
    exec_sc = AgentExecutor(agent=agent, tools=tools, memory=mem_sc, verbose=False, max_iterations=4)
    res = exec_sc.invoke({'input': sc['input']})
    pasos = res.get('intermediate_steps', [])
    herramientas = [p[0].tool for p in pasos] if pasos else ['ninguna (respuesta directa)']
    print(f'Decision: uso herramienta(s) → {herramientas}')
    print(f'Respuesta: {res["output"][:200]}')

---
## IL2.4 — Documentación Técnica (IE7, IE8)

### IE7 — Diagrama de Orquestación de Componentes

```
╔══════════════════════════════════════════════════════════════════════╗
║          ORQUESTACIÓN DEL AGENTE TCG — Empresa Trade SPA            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║   ┌───────────┐   input    ┌─────────────────────────────────────┐  ║
║   │  USUARIO  │──────────▶│           AGENTEXECUTOR             │  ║
║   └───────────┘           │  ┌──────────────────────────────┐   │  ║
║         ▲                 │  │    create_openai_tools_agent  │   │  ║
║         │ output          │  │    GPT-4o-mini  temp=0.3      │   │  ║
║         │                 │  └──────────────┬───────────────┘   │  ║
║         │                 │                 │ decide tool        │  ║
║         │                 │  ┌──────────────▼───────────────┐   │  ║
║         │                 │  │       TOOL SELECTOR          │   │  ║
║         │                 │  └──┬──────────┬────────────┬───┘   │  ║
║         │                 │     │          │            │        │  ║
║         │                 │  ┌──▼──┐    ┌──▼──┐    ┌───▼──┐    │  ║
║         │                 │  │ T1  │    │ T2  │    │  T3  │    │  ║
║         │                 │  │precio│   │ RAG │    │valor │    │  ║
║         │                 │  │carta│    │ TCG │    │colec.│    │  ║
║         │                 │  └──┬──┘    └──┬──┘    └───┬──┘    │  ║
║         │                 │     └──────────┴────────────┘        │  ║
║         │                 │                 │ tool result         │  ║
║         │                 │  ┌──────────────▼───────────────┐   │  ║
║         │                 │  │   KNOWLEDGE BASE TCG (12 docs)│   │  ║
║         │                 │  └──────────────────────────────┘   │  ║
║         │                 │                                      │  ║
║         │                 │  ┌──────────────────────────────┐   │  ║
║         │                 │  │  ConversationBufferWindowMem │   │  ║
║         │                 │  │  k=5 (corto plazo)           │   │  ║
║         │                 │  └──────────────────────────────┘   │  ║
║         │                 └────────────────────┬────────────────┘  ║
║         └────────────────────────────────────── ┘                  ║
╚══════════════════════════════════════════════════════════════════════╝
```

### IE8 — Justificación de Elección de Componentes

| Componente | Elección | Justificación técnica |
|---|---|---|
| **LLM** | GPT-4o-mini | Latencia baja (~2-4s), costo reducido, suficiente para dominio TCG estructurado |
| **Framework** | LangChain `create_openai_tools_agent` | Manejo automático del ciclo ReAct, soporte nativo para múltiples herramientas, extensible |
| **Herramientas** | `@tool` decorator | Inferencia automática de schema desde docstring, reduce boilerplate |
| **Memoria corto plazo** | `ConversationBufferWindowMemory(k=5)` | Ventana deslizante controla uso de tokens; k=5 cubre flujos típicos de consulta TCG |
| **Memoria largo plazo** | RAG sobre knowledge base | Proporciona hechos específicos del dominio sin dependencia de parámetros del LLM |
| **Planificación** | Plan-and-Execute con LLM planner | Separa razonamiento estratégico de ejecución táctica; mejora coherencia en tareas multi-paso |
| **Temperatura** | 0.3 | Respuestas consistentes para datos de precios; reduce alucinaciones en información factual |

In [ ]:
# IE7 - Resumen de la arquitectura del agente en código

resumen = {
    'framework': 'LangChain AgentExecutor + create_openai_tools_agent',
    'llm': 'GPT-4o-mini via GitHub Models API (Azure Inference)',
    'herramientas': [
        'consultar_precio_carta - busqueda directa en KB',
        'buscar_informacion_tcg - RAG lexico sobre 12 docs',
        'calcular_valor_coleccion - razonamiento sobre multiples cartas',
    ],
    'memoria_corto_plazo': 'ConversationBufferWindowMemory(k=5)',
    'memoria_largo_plazo': 'Knowledge Base TCG (12 documentos especializados)',
    'planificacion': 'Plan-and-Execute con LLM planner + AgentExecutor',
    'temperatura': 0.3,
    'max_iteraciones': 5,
}

print('RESUMEN ARQUITECTURA — AGENTE TCG EP2')
print('=' * 50)
for k, v in resumen.items():
    if isinstance(v, list):
        print(f'{k}:')
        for item in v: print(f'  - {item}')
    else:
        print(f'{k}: {v}')

---
## Conclusión (IE10)

### Logros técnicos del Agente TCG EP2

El agente implementado demuestra los cuatro indicadores de logro de IL2:

1. **IL2.1 — Agente funcional**: `AgentExecutor` con 3 herramientas especializadas que ejecuta funciones de consulta, recuperación RAG y razonamiento de forma autónoma, seleccionando la herramienta adecuada sin intervención humana.

2. **IL2.2 — Memoria**: `ConversationBufferWindowMemory(k=5)` asegura continuidad en flujos prolongados (demostrado con conversación de 5 turnos donde el agente recuerda el nombre del usuario). La knowledge base TCG actúa como memoria semántica de largo plazo.

3. **IL2.3 — Planificación**: El esquema Plan-and-Execute genera un plan explícito antes de ejecutar tareas complejas. Los 4 escenarios de toma de decisiones demuestran comportamiento adaptativo: el agente usa herramientas distintas según el tipo de consulta y declina apropiadamente consultas fuera del dominio TCG.

4. **IL2.4 — Documentación**: El diagrama de orquestación, la tabla de justificación de componentes y este README documentan completamente la arquitectura, decisiones de diseño y flujo de trabajo del sistema.

### Conexión con EP1

| EP1 (LLM + RAG) | EP2 (Agente) |
|---|---|
| Chatbot con prompt engineering | Agente con tool selection autónoma |
| RAG como arquitectura central | RAG como herramienta del agente |
| Memoria conversacional básica | Memoria de ventana deslizante |
| Evaluación con métricas RAGAS | Planificación y decisión adaptativa |

### Limitaciones y Trabajo Futuro
- Migrar memoria de largo plazo a FAISS con embeddings semánticos reales
- Integrar herramienta de búsqueda en TCGPlayer/Cardmarket API para precios en tiempo real
- Implementar orquestación multi-agente con CrewAI (Investigador + Valorador + Asesor)
- Añadir persistencia de memoria en Redis para usuarios recurrentes